
# TP 5 — Régularisation

**Objectifs de la séance.**
- Provoquer volontairement du sur-apprentissage (peu de données, modèle surdimensionné).
- Le corriger, un levier à la fois puis combinés : weight decay, dropout, batch normalization, early stopping.
- Découvrir les callbacks `EarlyStopping` et `ModelCheckpoint` de `training_toolbox`.

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

from training_toolbox import Trainer, EarlyStopping, ModelCheckpoint, accuracy

torch.manual_seed(0)


## Données : MNIST, volontairement sous-échantillonné

Pour provoquer facilement du sur-apprentissage, on entraîne sur un **tout petit** sous-ensemble de MNIST (500 images) : un modèle même modeste a largement la capacité d'« apprendre par cœur » ces 500 exemples plutôt que d'apprendre à généraliser. Le jeu de validation, lui, reste de taille normale.

In [ ]:

from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])

mnist_train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
mnist_test_full = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

torch.manual_seed(0)
small_indices = torch.randperm(len(mnist_train_full))[:500]
mnist_train_small = Subset(mnist_train_full, small_indices)

train_loader = DataLoader(mnist_train_small, batch_size=32, shuffle=True)
val_loader = DataLoader(mnist_test_full, batch_size=256, shuffle=False)

print("Taille du jeu d'entraînement :", len(mnist_train_small))

In [ ]:

def plot_curves(history, metric):
    plt.figure()
    plt.plot(history[f"train_{metric}"], label=f"train_{metric}")
    if history.get(f"val_{metric}"):
        plt.plot(history[f"val_{metric}"], label=f"val_{metric}")
    plt.xlabel("epoch"); plt.legend(); plt.title(metric)
    plt.show()


## Partie 1 — Provoquer le sur-apprentissage

**Question 1.1.** Définissez un MLP `BigMLP` volontairement surdimensionné pour 500 exemples (par exemple 3 couches cachées de 512 neurones, activations ReLU, 10 sorties). Entraînez-le avec le `Trainer` (`nn.CrossEntropyLoss()`, `torch.optim.Adam`, `metrics={"acc": accuracy}`) pendant 40 epochs sur `train_loader`/`val_loader`, en conservant l'historique dans `history_overfit`.

In [ ]:
class BigMLP(nn.Module):
    def __init__(self, n_features=784, hidden_size=512, n_classes=10):
        super().__init__()
        # TODO : 3 couches cachées de taille hidden_size (ReLU entre chaque), puis une sortie
        #        à n_classes neurones (sans activation)
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_classes),
        )

    def forward(self, x):
        # TODO
        return self.net(x)


# TODO : instancier BigMLP, un optimizer Adam, un Trainer, et appeler .fit(train_loader, val_loader, epochs=40)
#        en conservant le résultat dans history_overfit
overfit_model = BigMLP()
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=1e-3)
overfit_trainer = Trainer(overfit_model, overfit_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy})
history_overfit = overfit_trainer.fit(train_loader, val_loader, epochs=40)


**Question 1.2.** Tracez `loss` puis `acc` (train vs val) en fonction des epochs, avec `plot_curves`. Repérez à l'œil le moment où les courbes train/val commencent à diverger.

In [ ]:

plot_curves(history_overfit, "loss")
plot_curves(history_overfit, "acc")

# TODO : BigMLP neuf, optimizer Adam avec weight_decay=1e-3, entraînement 40 epochs -> history_wd
#        puis plot_curves(history_wd, "loss") et plot_curves(history_wd, "acc")
wd_model = BigMLP()
wd_optimizer = torch.optim.Adam(wd_model.parameters(), lr=1e-3, weight_decay=1e-3)
wd_trainer = Trainer(wd_model, wd_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy})
history_wd = wd_trainer.fit(train_loader, val_loader, epochs=40)

plot_curves(history_wd, "loss")
plot_curves(history_wd, "acc")

In [ ]:

# TODO : BigMLP neuf, optimizer Adam avec weight_decay=1e-3, entraînement 40 epochs -> history_wd
#        puis plot_curves(history_wd, "loss") et plot_curves(history_wd, "acc")


## Partie 3 — Dropout

**Question 3.1.** Définissez `BigMLPDropout`, identique à `BigMLP` mais avec une couche `nn.Dropout(p=0.3)` après chaque activation ReLU (rappel : le dropout n'est actif qu'en mode entraînement — `model.train()`/`model.eval()`, déjà géré par le `Trainer`). Entraînez-le (sans weight decay, pour isoler l'effet du dropout) et comparez à `history_overfit`.

In [ ]:
class BigMLPDropout(nn.Module):
    def __init__(self, n_features=784, hidden_size=512, n_classes=10, p=0.3):
        super().__init__()
        # TODO : mêmes couches que BigMLP, avec un nn.Dropout(p) après chaque ReLU
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, n_classes),
        )

    def forward(self, x):
        # TODO
        return self.net(x)


# TODO : entraîner BigMLPDropout (mêmes hyperparamètres que la partie 1, pas de weight decay)
#        -> history_dropout, puis tracer les courbes
dropout_model = BigMLPDropout()
dropout_optimizer = torch.optim.Adam(dropout_model.parameters(), lr=1e-3)
dropout_trainer = Trainer(dropout_model, dropout_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy})
history_dropout = dropout_trainer.fit(train_loader, val_loader, epochs=40)

plot_curves(history_dropout, "loss")
plot_curves(history_dropout, "acc")


## Partie 4 — Batch normalization

La *batch normalization* normalise (moyenne 0, variance 1, sur le mini-batch courant) la sortie d'une couche avant de lui appliquer l'activation, ce qui stabilise et accélère souvent l'entraînement ; elle a aussi un effet régularisant, plus modeste, en introduisant du bruit lié à l'estimation de moyenne/variance sur un mini-batch.

**Question 4.1.** Définissez `BigMLPBatchNorm`, identique à `BigMLP` mais avec une couche `nn.BatchNorm1d(hidden_size)` insérée entre chaque couche linéaire cachée et son activation ReLU. Entraînez-le (sans dropout ni weight decay, pour isoler l'effet) et comparez à `history_overfit`.

In [ ]:
class BigMLPBatchNorm(nn.Module):
    def __init__(self, n_features=784, hidden_size=512, n_classes=10):
        super().__init__()
        # TODO : mêmes couches que BigMLP, avec un nn.BatchNorm1d(hidden_size) entre chaque
        #        couche linéaire cachée et sa ReLU
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_classes),
        )

    def forward(self, x):
        # TODO
        return self.net(x)


# TODO : entraîner BigMLPBatchNorm (mêmes hyperparamètres que la partie 1)
#        -> history_bn, puis tracer les courbes
bn_model = BigMLPBatchNorm()
bn_optimizer = torch.optim.Adam(bn_model.parameters(), lr=1e-3)
bn_trainer = Trainer(bn_model, bn_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy})
history_bn = bn_trainer.fit(train_loader, val_loader, epochs=40)

plot_curves(history_bn, "loss")
plot_curves(history_bn, "acc")


## Partie 5 — Early stopping et sauvegarde du meilleur modèle

`training_toolbox` fournit deux callbacks à passer au `Trainer` :

- `EarlyStopping(patience=..., monitor="val_loss")` : arrête l'entraînement si `val_loss` ne s'améliore plus pendant `patience` epochs consécutives ;
- `ModelCheckpoint(path=..., monitor="val_loss")` : sauvegarde les poids du modèle (via `torch.save`) à chaque fois que `val_loss` s'améliore.

**Question 5.1.** Réentraînez un `BigMLP` neuf pendant un grand nombre d'epochs (par exemple 100 — on compte sur l'early stopping pour arrêter avant la fin), avec `callbacks=[EarlyStopping(patience=5), ModelCheckpoint("best_model.pt")]`. Combien d'epochs l'entraînement a-t-il effectivement duré ?

In [ ]:
# TODO : BigMLP neuf, Trainer avec callbacks=[EarlyStopping(patience=5), ModelCheckpoint("best_model.pt")],
#        .fit(train_loader, val_loader, epochs=100) -> regarder len(history["train_loss"])
es_model = BigMLP()
es_optimizer = torch.optim.Adam(es_model.parameters(), lr=1e-3)
es_trainer = Trainer(
    es_model, es_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy},
    callbacks=[EarlyStopping(patience=5), ModelCheckpoint("best_model.pt")],
)
history_es = es_trainer.fit(train_loader, val_loader, epochs=100)

print("Nombre d'epochs effectuées :", len(history_es["train_loss"]))


**Question 5.2.** Rechargez les poids sauvegardés dans un modèle neuf de même architecture (`model.load_state_dict(torch.load("best_model.pt"))`), et évaluez-le sur `val_loader` avec `trainer.evaluate(...)`. Comparez à la dernière valeur de `val_loss` de l'historique : le modèle sauvegardé est-il bien celui de la meilleure epoch, et non celui de la dernière ?

In [ ]:
# TODO : charger un BigMLP neuf, model.load_state_dict(torch.load("best_model.pt")),
#        construire un Trainer autour et appeler .evaluate(val_loader) ; comparer au dernier val_loss
#        de l'historique de la question précédente
reloaded_model = BigMLP()
reloaded_model.load_state_dict(torch.load("best_model.pt"))
reloaded_optimizer = torch.optim.Adam(reloaded_model.parameters(), lr=1e-3)
reloaded_trainer = Trainer(reloaded_model, reloaded_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy})

reloaded_stats = reloaded_trainer.evaluate(val_loader)
print("val_loss (modèle rechargé, meilleure epoch) :", reloaded_stats["loss"])
print("val_loss (dernière epoch de l'historique)   :", history_es["val_loss"][-1])


## Partie 6 — Tout combiner

**Question 6.1.** Définissez un dernier modèle combinant dropout **et** batch normalization, entraîné avec weight decay **et** les callbacks `EarlyStopping`/`ModelCheckpoint`. Comparez ses courbes finales à `history_overfit` : l'écart train/val s'est-il réduit ?

In [ ]:
class BigMLPRegularized(nn.Module):
    def __init__(self, n_features=784, hidden_size=512, n_classes=10, p=0.3):
        super().__init__()
        # TODO : couches linéaires + BatchNorm1d + ReLU + Dropout(p), empilées, puis sortie finale
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, n_classes),
        )

    def forward(self, x):
        # TODO
        return self.net(x)


# TODO : entraîner BigMLPRegularized avec weight_decay et callbacks=[EarlyStopping, ModelCheckpoint]
#        -> history_combined, puis comparer les courbes à history_overfit
combined_model = BigMLPRegularized()
combined_optimizer = torch.optim.Adam(combined_model.parameters(), lr=1e-3, weight_decay=1e-3)
combined_trainer = Trainer(
    combined_model, combined_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy},
    callbacks=[EarlyStopping(patience=5), ModelCheckpoint("best_combined.pt")],
)
history_combined = combined_trainer.fit(train_loader, val_loader, epochs=100)

plot_curves(history_overfit, "loss")
plot_curves(history_combined, "loss")
plot_curves(history_overfit, "acc")
plot_curves(history_combined, "acc")


**Questions.**
- Le sur-apprentissage a-t-il complètement disparu, ou seulement diminué ? Est-ce surprenant avec seulement 500 exemples d'entraînement ?
- Parmi les leviers testés, lequel a eu l'effet le plus visible sur ce problème précis ? Ce classement serait-il nécessairement le même sur un autre dataset/une autre architecture ?

_Le sur-apprentissage n'a pas complètement disparu, il est seulement fortement atténué : avec seulement 500 exemples d'entraînement, un MLP de plusieurs centaines de milliers de paramètres reste capable de mémoriser une bonne partie du jeu d'entraînement même régularisé — ce n'est pas surprenant, le nombre de paramètres restant très supérieur au nombre d'exemples. Sur ce problème précis, l'early stopping (combiné au dropout) est en général le levier avec l'effet le plus visible sur l'écart train/val, puisqu'il stoppe directement l'entraînement au moment où la validation se dégrade ; le weight decay et surtout la batch normalization (dont l'effet régularisant est plus indirect, pensé avant tout pour stabiliser l'optimisation) ont ici un impact plus modeste. Ce classement dépend fortement du dataset et de l'architecture : sur un plus gros modèle convolutif par exemple, le poids relatif de la batch norm et du dropout peut être très différent._


## Bilan

Vous disposez maintenant de tout l'attirail de régularisation standard (weight decay, dropout, batch normalization, early stopping) et savez les combiner via le `Trainer`. La séance 6 est un mini-projet qui mobilise l'ensemble des séances 3 à 5 sur un jeu de données tabulaire réaliste, avec un ingrédient nouveau : les variables catégorielles.